In [1]:
notes = '''This copy is featured with initial gamma = 0.3.\n
We set f_opt_kh = 1, use_a = True, and L1_regularization = 0, so it is not working.\n
We use warmup_exponential_decay_schedule for the learning rate.\n
We use normal sampling to optimize k_h with sigma = 0.2.
In this copy, we train 50000 epoches. '''
print(notes)

In [2]:
import sys
# sys.path.insert(0, '../../Utilities/')

# from jax.example_libraries import optimizers
import jax.numpy as jnp
import numpy as np
from jax import grad, jit, vmap, value_and_grad, hessian
from jax import random
from functools import partial
import flax.linen as nn
import jax
import optax
import pickle
from datetime import datetime
import os
import numpy as np
from scipy import sparse
import time
from jax import grad, hessian, jit, vmap, pmap
from jax.tree_util import tree_leaves
from datetime import datetime
import matplotlib.pyplot as plt
import pickle
import flax.linen as nn
import optax
import optax.tree_utils as otu

from minmax_opt_class_mgnn import MinMaxOptProb

import os
os.environ['ENABLE_PJRT_COMPATIBILITY'] = '1'

jax.config.update('jax_platform_name', 'cpu')
os.environ['OMP_NUM_THREADS'] = str(os.cpu_count())

In [3]:
eps0 = 1e-4
def smooth_sign(x, eps = eps0):
    return 2*jax.nn.sigmoid(x/eps)-1

# def u0(x):
#     return 0.5 * (smooth_sign(x - 0.4) - smooth_sign(x - 0.6))

# Define boundary and initial conditions
def u0(x):
    # return 2*smooth_sign(x) - 1
    return -jnp.sign(x)
    # return -jnp.sin(jnp.pi*x)

# Boundary at x=-1
def u1(x):
    return 1

# Boundary at x=1
def u2(x):
    return -1

def res_visc(u_quad, u_t, u_x, u_xx, nu, u_exact=0):
    return u_t + (u_quad**2)*u_x - nu*u_xx

def res(u_quad, u_t, u_x):
    return u_t + (u_quad**2)*u_x

# Initialize the optimization problem
optProb = MinMaxOptProb(
    N_t=4,
    N_x=4,
    N_bc=100,
    N_ic=400,
    t_min=0, t_max=0.5, x_min=-1, x_max=1,
    project='minmax_mgnn_burgers_inti_sin',
    notes=notes,
    L1_regularization = 0.0,
    sampling_method = 'normal',
    hidden_dims_list=[(40, 40, 40, 40)],
    init_lrs=[1e-3, 1e-3, 1e-3, 1e-3], grade_epochs=[50000, 50000, 50000, 50000], decay_rates=[1e-3, 1e-3, 1e-3, 1e-3], init_lr_stage2=1e-3, epoch_stage2=100000, decay_rate_stage2=1e-3,
    seed=0, sigma=0.2,
    u0=u0, u1=u1, u2=u2,
    nu=0.01,
    res_visc=res_visc,
    res=res,
    N_perturb = 10000,
    f_adaptive_weight = 10000,
    f_opt_kh=1,
    total_grade=1, k=0,
)

W0000 00:00:1752685667.748967 10486232 mps_client.cc:510] WARNING: JAX Apple GPU support is experimental and not all JAX functionality is correctly supported!
I0000 00:00:1752685667.757482 10486232 service.cc:145] XLA service 0x600000c3de00 initialized for platform METAL (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1752685667.757491 10486232 service.cc:153]   StreamExecutor device (0): Metal, <undefined>
I0000 00:00:1752685667.758482 10486232 mps_client.cc:406] Using Simple allocator.
I0000 00:00:1752685667.758488 10486232 mps_client.cc:384] XLA backend will use up to 103078739968 bytes on device 0 for SimpleAllocator.


Metal device set to: Apple M4 Max


In [4]:
optProb.train_process_stage1()


Starting Train Stage with MGDL

Stage 1: Sequential Grade Training

Training Grade 0
Nt: 32, losses_all shape: (32, 32)
Nt: 32, loss_abs shape: (32,)
best_coeffs_all shape: (32, 3), perturbed_coeffs_all shape: (32, 10000, 3)
self.pwp.coeffs shape: (32, 3)
Stage 1, Iter 0, Gamma: 0.30000000
Nt: 32, losses_all shape: (32, 32)
Nt: 32, loss_abs shape: (32,)
best_coeffs_all shape: (32, 3), perturbed_coeffs_all shape: (32, 10000, 3)
Grade 0, Iter 0, loss_eqn: 0.07860289, loss_eqn_from_kh: -0.13542566, loss_eqn_abs: 0.07860294, loss_ibc: 2.44860315
Nt: 32, losses_all shape: (32, 32)
Nt: 32, loss_abs shape: (32,)
best_coeffs_all shape: (32, 3), perturbed_coeffs_all shape: (32, 10000, 3)
Grade 0, Iter 1, loss_eqn: 0.07717869, loss_eqn_from_kh: 0.17581326, loss_eqn_abs: 0.07717866, loss_ibc: 2.44452620
Nt: 32, losses_all shape: (32, 32)
Nt: 32, loss_abs shape: (32,)
best_coeffs_all shape: (32, 3), perturbed_coeffs_all shape: (32, 10000, 3)
Grade 0, Iter 2, loss_eqn: 0.07450163, loss_eqn_from_kh

KeyboardInterrupt: 

wandb: Currently logged in as: haoli1 to https://api.wandb.ai. Use `wandb login --relogin` to force relogin



Stage 1: Sequential Grade Training

Training Grade 0
tri_idx: Traced<ShapedArray(int32[])>with<BatchTrace> with
  val = Array([  0,   1,   2,   3,   4,   5,   6,   7,   8,   9,  10,  11,  12,
        13,  14,  15,  16,  17,  18,  19,  20,  21,  22,  23,  24,  25,
        26,  27,  28,  29,  30,  31,  32,  33,  34,  35,  36,  37,  38,
        39,  40,  41,  42,  43,  44,  45,  46,  47,  48,  49,  50,  51,
        52,  53,  54,  55,  56,  57,  58,  59,  60,  61,  62,  63,  64,
        65,  66,  67,  68,  69,  70,  71,  72,  73,  74,  75,  76,  77,
        78,  79,  80,  81,  82,  83,  84,  85,  86,  87,  88,  89,  90,
        91,  92,  93,  94,  95,  96,  97,  98,  99, 100, 101, 102, 103,
       104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116,
       117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129,
       130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142,
       143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155,
       156, 157

ValueError: Incompatible shapes for broadcasting: shapes=[(6144, 3), (6144,)]

In [ ]:
optProb.train_process_stage2()

In [ ]:
from my_helpers import plot_u_theta
plot_u_theta(optProb.model, M_t=500, M_x=1000, t_min=0.0, t_max=1, x_min=-1.0, x_max=1.0, ts=[0.1*i for i in range(11)])  # Plot initial model state


In [ ]:
params = optProb.model.mlps[0].params
optProb.loss_fn_ibc(params)

In [8]:
optProb.pwp.triangles.shape

In [ ]:
list(params)[0]

In [ ]:
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
model_path = os.path.join('..', 'data', f'train_model_{timestamp}.pkl')

In [ ]:
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
model_path = os.path.join('data', f'train_model_{timestamp}.pkl')
optProb.save_model(model_path)
print(f"Train model saved to {model_path}")

In [ ]:
optProb.save_model(model_path)

In [ ]:
os.makedirs(os.path.dirname(model_path), exist_ok=True)

In [ ]:
model_path = os.path.join('data', f'train_model_{timestamp}.pkl')

In [ ]:
import matplotlib.pyplot as plt
import jax.numpy as jnp

def plot_u_theta(model, params=None, M_t=100, M_x=100, t_min=0.0, t_max=1.0, x_min=0.0, x_max=1.0):
    """
    Plot the neural network function u_theta over a specified domain [t_min, t_max] x [x_min, x_max],
    and include a 1D plot of u_theta at t = t_max.

    Args:
        model: Flax model instance representing u_theta.
        params: Parameters (pytree) of the neural network.
        M_t: Number of grid points in the t-direction (default: 100).
        M_x: Number of grid points in the x-direction (default: 100).
        t_min: Minimum value for t (default: 0.0).
        t_max: Maximum value for t (default: 1.0).
        x_min: Minimum value for x (default: 0.0).
        x_max: Maximum value for x (default: 1.0).
    """
    # Create a grid for t and x over the specified ranges
    t_grid = jnp.linspace(t_min, t_max, M_t)
    x_grid = jnp.linspace(x_min, x_max, M_x)
    t, x = jnp.meshgrid(t_grid, x_grid, indexing='ij')  # Shapes: (M_t, M_x)

    # Flatten the grid for batch evaluation
    tx_points = jnp.stack([t.ravel(), x.ravel()], axis=1)  # Shape: (M_t * M_x, 2)

    # Evaluate u_theta on the grid points
    if params is None:
        u_values = model.forward(tx_points)
    else:
        u_values = model(params, tx_points)  # Shape: (M_t * M_x, 1)
    u_values = u_values.reshape(M_t, M_x)  # Reshape to (M_t, M_x)

    # Create figure with two subplots
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8))

    # 2D color mesh plot
    pcm = ax1.pcolormesh(x_grid, t_grid, u_values, shading='nearest', cmap='viridis')
    fig.colorbar(pcm, ax=ax1, label='$u_\\theta(t, x)$')
    ax1.set_xlabel('x')
    ax1.set_ylabel('t')
    ax1.set_title('Neural Network Function $u_\\theta$')

    # 1D plot at t = t_max
    ax2.plot(x_grid, u_values[-1, :])
    ax2.set_xlabel('x')
    ax2.set_ylabel('$u_\\theta(t_{max}, x)$')
    ax2.set_title(f'Neural Network Function at t = {t_max}')

    # Adjust layout and display
    plt.tight_layout()
    plt.show()

In [ ]:
plot_u_theta(optProb.model, M_t=500, M_x=1000, t_min=0.0, t_max=1, x_min=-1.0, x_max=1.0)  # Plot initial model state


In [ ]:
print(os.environ.get('ENABLE_PJRT_COMPATIBILITY'))

In [26]:
A_all.shape

In [25]:
u_values_all.shape

In [10]:
params = optProb.model.mlps[0].params
Nt = optProb.pwp.N_elements
vertices_all = optProb.pwp.triangles
u_values_all = optProb.u_eval(params, vertices_all.reshape(-1, 2)).reshape(Nt, 3)
A_all = jnp.stack([jnp.ones((Nt, 3)), vertices_all[:, :, 0], vertices_all[:, :, 1]], axis=-1)
init_coeffs_all = jnp.linalg.solve(A_all, u_values_all[..., None]).squeeze(-1)
perturbations_all = vmap(lambda k: 0.2 * jax.random.normal(k, (1000, 3)))(keys)


In [11]:
tri_idx = 0
quad_points = optProb.pwp.quad_points_tri[tri_idx]
quad_weights = optProb.pwp.quad_weights_tri[tri_idx]
p_coeffs_all = perturbations_all[tri_idx]
k_h_quad_all = (p_coeffs_all[:, 0][:, None] + 
                p_coeffs_all[:, 1][:, None] * quad_points[:, 0] + 
                p_coeffs_all[:, 2][:, None] * quad_points[:, 1])

u_quad = optProb.u_eval(params, quad_points)
grad_u_tx = optProb.grad_u(params, quad_points)
u_t, u_x = grad_u_tx[:, 0], grad_u_tx[:, 1]
residual = optProb.res(u_quad, u_t, u_x)

In [12]:
current_key = jax.random.PRNGKey(0)
key, current_key = jax.random.split(current_key)
keys = jax.random.split(key, Nt)
integrand_all = residual[None, :] * jnp.sign(u_quad[None, :] - k_h_quad_all)

In [58]:
losses_all = vmap(optProb.loss_kh_per_elem, in_axes=(0, 0, None))(jnp.arange(Nt), perturbed_coeffs_all, params)
best_indices = jnp.argmax(losses_all, axis=1)
loss_eqn = jnp.sum(losses_all[:, best_indices])

In [13]:
residual

Array([0.11973645, 0.129627  , 0.10402286, 0.129627  , 0.12834598,
       0.11847676, 0.10402286, 0.11847676, 0.08497735, 0.129627  ,
       0.11847676, 0.10402286], dtype=float32)

In [46]:
@vmap
def gen_pert(key):
    return 0.2 * jax.random.normal(key, (10000, 3))

In [86]:
perturbations_all = gen_pert(keys)
perturbed_coeffs_all = init_coeffs_all[:, None, :] + perturbations_all

In [87]:
perturbed_coeffs_all.shape

(512, 10000, 3)

5.45 ms ± 217 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [62]:
losses_all.shape

In [72]:
pwp.quad_weights

Array([ 0.00023935, -0.00023935,  0.00023935, ...,  0.00023935,
        0.00023935, -0.00023935], dtype=float32)

In [73]:
plt.plot(losses_all[0,:100])

In [122]:
residual

Array([0.11973645, 0.1259087 , 0.11798877, 0.1259087 , 0.129627  ,
       0.12444635, 0.11798877, 0.12444635, 0.11617592, 0.1259087 ,
       0.12444635, 0.11798877], dtype=float32)

In [129]:
jnp.sum(quad_weights[None, :] * integrand_all, axis=1)

Array([ 0.00023935, -0.00023935,  0.00023935, ...,  0.00023935,
        0.00023935, -0.00023935], dtype=float32)

In [84]:
optProb.u_eval(params, quad_points)

Array([0.09147714, 0.09534946, 0.08824384, 0.09534946, 0.09938046,
       0.09206421, 0.08824384, 0.09206421, 0.08498695, 0.09534946,
       0.09206421, 0.08824384], dtype=float32)

In [115]:
grad_u_tx

Array([[ 0.12059903, -0.1030803 ],
       [ 0.12686068, -0.10471053],
       [ 0.11879742, -0.10384681],
       [ 0.12686068, -0.10471053],
       [ 0.13067424, -0.10603414],
       [ 0.12534094, -0.105546  ],
       [ 0.11879742, -0.10384681],
       [ 0.12534094, -0.105546  ],
       [ 0.11693138, -0.1045938 ],
       [ 0.12686068, -0.10471053],
       [ 0.12534094, -0.105546  ],
       [ 0.11879742, -0.10384681]], dtype=float32)

In [88]:
params = optProb.model.mlps[0].params
optProb.pwp.coeffs, optProb.key, loss_eqn_from_kh, loss_eqn_abs = optProb.maximize_kh(params, optProb.key)
print(loss_eqn_from_kh)
print(loss_eqn_abs)

Nt: 32, losses_all shape: (32, 32)
Nt: 32, loss_abs shape: (32,)
best_coeffs_all shape: (32, 3), perturbed_coeffs_all shape: (32, 10000, 3)
-0.17287518
0.015171622


In [89]:
optProb.loss_fn_elem_int(params)

Array(0.01406719, dtype=float32)

In [127]:
(integrand_all * quad_weights[None, :]).shape

(10000, 12)

In [177]:
optProb.N_perturb=100000

In [176]:
quad_weights@jnp.ones(12)

Array(0.00195312, dtype=float32)

In [7]:
optProb.pwp.quad_weights.flatten().shape

(384,)

In [54]:
params = optProb.model.mlps[0].params
quad_points = optProb.pwp.quad_points
quad_weights = optProb.pwp.quad_weights
u_quad = optProb.u_eval(params, quad_points)
grad_u_tx = optProb.grad_u(params, quad_points)
u_t, u_x = grad_u_tx[:, 0], grad_u_tx[:, 1]            
residual = optProb.res(u_quad, u_t, u_x)
p_coeffs_quad = optProb.pwp.coeffs[optProb.pwp.element_indices]
k_h_quad = p_coeffs_quad[:, 0] + p_coeffs_quad[:, 1] * quad_points[:, 0] + p_coeffs_quad[:, 2] * quad_points[:, 1]
integrand = residual * jnp.sign(u_quad - k_h_quad)
interior_loss = jnp.dot(quad_weights, integrand)

In [55]:
chosen_idx = (optProb.pwp.element_indices == 1)

In [56]:
integrand = residual * jnp.sign(u_quad - k_h_quad)
jnp.dot(quad_weights[chosen_idx], integrand[chosen_idx])

Array(7.014592e-05, dtype=float32)

In [44]:
integrand = residual * jnp.sign(u_quad - k_h_quad)
jnp.dot(quad_weights[chosen_idx], integrand[chosen_idx])

Array(7.014592e-05, dtype=float32)

In [48]:
jnp.sum(integrand[chosen_idx])

Array(0.02693603, dtype=float32)

In [47]:
quad_weights[chosen_idx]

Array([0.00260417, 0.00260417, 0.00260417, 0.00260417, 0.00260417,
       0.00260417, 0.00260417, 0.00260417, 0.00260417, 0.00260417,
       0.00260417, 0.00260417], dtype=float32)

In [51]:
interior_loss

Array(0.01406719, dtype=float32)

In [25]:
optProb.pwp.element_indices

Array([ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  1,  1,  1,  1,  1,
        1,  1,  1,  1,  1,  1,  1,  2,  2,  2,  2,  2,  2,  2,  2,  2,  2,
        2,  2,  3,  3,  3,  3,  3,  3,  3,  3,  3,  3,  3,  3,  4,  4,  4,
        4,  4,  4,  4,  4,  4,  4,  4,  4,  5,  5,  5,  5,  5,  5,  5,  5,
        5,  5,  5,  5,  6,  6,  6,  6,  6,  6,  6,  6,  6,  6,  6,  6,  7,
        7,  7,  7,  7,  7,  7,  7,  7,  7,  7,  7,  8,  8,  8,  8,  8,  8,
        8,  8,  8,  8,  8,  8,  9,  9,  9,  9,  9,  9,  9,  9,  9,  9,  9,
        9, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 11, 11, 11, 11,
       11, 11, 11, 11, 11, 11, 11, 11, 12, 12, 12, 12, 12, 12, 12, 12, 12,
       12, 12, 12, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 14, 14,
       14, 14, 14, 14, 14, 14, 14, 14, 14, 14, 15, 15, 15, 15, 15, 15, 15,
       15, 15, 15, 15, 15, 16, 16, 16, 16, 16, 16, 16, 16, 16, 16, 16, 16,
       17, 17, 17, 17, 17, 17, 17, 17, 17, 17, 17, 17, 18, 18, 18, 18, 18,
       18, 18, 18, 18, 18